# ESMFold Structure Prediction

![ESMFold Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/esmfold/hero.png)

This notebook demonstrates protein structure prediction using ESMFold. We cover a single-sequence prediction with result inspection and visualization, followed by a batch prediction over multiple sequences. ESMFold runs without multiple sequence alignments, making it fast enough for iterative design workflows where hundreds of candidate sequences need structural evaluation.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esmfold")
display_overview("esmfold")
display_docs_section("esmfold", "Background")

## Available tools

In [ ]:
display_available_tools("esmfold")

### `run_esmfold`

ESMFold predicts protein 3D structures directly from amino acid sequence without requiring multiple sequence alignments. It uses a large protein language model backbone to infer residue-level geometry, producing atomic coordinates along with per-residue pLDDT confidence scores and a global pTM score. The lack of an MSA step makes it substantially faster than alignment-dependent methods, which is especially valuable for screening large libraries of designed sequences.

In [ ]:
from proto_tools import ESMFoldInput, ESMFoldConfig, run_esmfold

In [ ]:
# Display input API reference
display_api_reference("esmfold", "input", "run_esmfold")

# Ubiquitin (76 residues): a small, well-ordered domain ESMFold folds with high confidence
ubiquitin_sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"

# Create input
inputs = ESMFoldInput(complexes=[ubiquitin_sequence])

In [ ]:
# Display config API reference
display_api_reference("esmfold", "config", "run_esmfold")

# Configure ESMFold
config = ESMFoldConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the prediction
result = run_esmfold(inputs, config)

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
# Display output API reference
display_api_reference("esmfold", "output", "run_esmfold")

ubiquitin_structure = result.structures[0]

# Print confidence metrics
print(f"Length:       {ubiquitin_structure.num_residues} residues")
print(f"Average pLDDT: {ubiquitin_structure.metrics.avg_plddt:.3f}")
print(f"pTM score:    {ubiquitin_structure.metrics.ptm:.3f}")

# Visualize predicted structure colored by pLDDT confidence
ubiquitin_structure.visualize()

### `run_esmfold_gradient`

The gradient tool runs ESMFold over a relaxed `(L, 20)` distribution for one or more designated target chains in a complex and returns the gradient of a weighted confidence loss (`pLDDT`, `pTM`, `pAE`) with respect to those input logits. The relaxed distribution is mixed into both ESM-2's word embeddings and ESMFold's own AA embedding, so the structure module's confidence heads can differentiate back to the input logits while the model parameters stay frozen. Use this as a structure-aware loss inside MCMC or gradient descent over relaxed protein sequences. Set `compute_gradient=False` for forward-only confidence scoring.

In [ ]:
from proto_tools import (
    ESMFoldGradientConfig,
    ESMFoldGradientInput,
    run_esmfold_gradient,
)
from proto_tools.utils import one_hot_protein_logits


In [ ]:
# Display input API reference
display_api_reference("esmfold", "input", "run_esmfold_gradient")

# Seed a relaxed target-chain distribution from a discrete sequence (sharpness=2.0
# yields a biased-but-not-saturated softmax target).
target_seq = "MKTAYIAKQR"
logits = one_hot_protein_logits(target_seq, sharpness=2.0)

# Each candidate bundles its relaxed state with the complex it is folded within
inputs = ESMFoldGradientInput(
    inputs={
        "logits": logits,
        "chains": [target_seq],        # length must equal len(logits) for every target chain
        "target_chain_indices": [0],   # which chains receive the relaxed distribution
    },
)


In [ ]:
# Display config API reference
display_api_reference("esmfold", "config", "run_esmfold_gradient")

# Configure the differentiable confidence pass
config = ESMFoldGradientConfig(
    loss_weights={"plddt": 1.0, "ptm": 0.5},   # weighted sum of confidence terms
    soft=1.0,                                  # 1.0 = pure softmax probabilities
    hard=0.0,                                  # set to 1.0 for a straight-through estimator
    compute_gradient=True,
    num_recycles=1,                            # keep recycles low for gradient passes
    device="cuda",                             # Change to "cpu" if no GPU available
)


In [ ]:
# Run one differentiable confidence pass
gradient_result = run_esmfold_gradient(inputs, config)


In [ ]:
# Display output API reference
display_api_reference("esmfold", "output", "run_esmfold_gradient")

# One result per candidate, each carrying its own predicted structure
candidate = gradient_result.results[0]

print(f"Weighted loss: {candidate.loss:.3f}")
print(f"Avg pLDDT:    {candidate.metrics['avg_plddt']:.3f}")
print(f"loss_plddt:   {candidate.metrics['loss_plddt']:.3f}")
if 'loss_ptm' in candidate.metrics:
    print(f"loss_ptm:     {candidate.metrics['loss_ptm']:.3f}")
assert candidate.gradient is not None  # backward pass enabled
print(f"Gradient shape: ({len(candidate.gradient)}, {len(candidate.gradient[0])})")

# The predicted structure is also returned, colored by pLDDT
candidate.structure.visualize()


## Batch Predictions

Protein design workflows routinely generate dozens to hundreds of candidate sequences that each require structural evaluation. ESMFold supports batch input natively, processing all sequences in a single call with automatic GPU batching for memory safety. The output contains one predicted structure per input sequence in the same order as the input list.

In [ ]:
# Four distinct, well-folded small proteins spanning different structural classes.
sequences = [
    "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",   # Protein G B1 domain (alpha/beta)
    "RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFKSAEDCMRTCGGA",  # BPTI (Kunitz fold)
    "TTCCPSIVARSNFNVCRLPGTPEAICATYTGCIIIPGATCPGDYAN",            # Crambin (disulfide-rich)
    "MLSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF",                       # Villin headpiece HP36 (helical)
]

# Create batch input
batch_input = ESMFoldInput(complexes=sequences)

In [ ]:
# Configure batch prediction
batch_config = ESMFoldConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the batch prediction
batch_result = run_esmfold(batch_input, batch_config)

In [ ]:
# Inspect results for all predicted structures
for i, structure in enumerate(batch_result.structures):
    print(f"Structure {i + 1}:")
    print(f"  Length:        {structure.num_residues} residues")
    print(f"  Average pLDDT: {structure.metrics.avg_plddt:.3f}")
    print(f"  pTM score:     {structure.metrics.ptm:.3f}")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to further computational workflows including molecular dynamics, docking, or inverse folding. The B-factor column in exported files contains the normalized pLDDT confidence scores (0.0-1.0), so coloring by B-factor in a viewer directly shows per-residue model confidence.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export single-sequence prediction to PDB
result.export(name="ubiquitin_structure", export_path=output_dir, file_format="pdb")

# Export batch predictions to PDB
batch_result.export(name="batch_structures", export_path=output_dir, file_format="pdb")